# 03 · Out-of-fold 평가 및 논문 비교
각 검증 케이스는 그 케이스를 학습하지 않은 fold의 최종 모델로 예측합니다.
학습 종료 시 저장되는 전체 볼륨 validation 예측을 사용합니다. 추가 앙상블/후처리는 적용하지 않습니다.
영역: kidney=(1,2,3), masses=(2,3), tumor=(2). NSD tolerance=2mm.
케이스별 → 영역별 → fold별 평균을 계산합니다. 양쪽 모두 빈 영역은 NaN으로 제외합니다.
논문의 정확한 코드/분할/집계 세부 조건과의 동일성이 확인되지 않은 부분은 보고서에 남습니다.
전체 5-fold를 선택하지 않으면 부분 결과입니다. fold 0 점수를 논문 5-fold 평균과 동등하게 해석하지 않습니다.

In [ ]:
from pathlib import Path
import sys, os, json, shlex, subprocess
sys.dont_write_bytecode = True
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nnunetv2").is_dir())
EXPERIMENT = REPO / "experiments/kits2023"
sys.path.insert(0, str(EXPERIMENT / "scripts"))
from workflow import read_config, paths, guard, environment, model_dir
CONFIG = EXPERIMENT / "config.server.json"
if not CONFIG.exists():
    CONFIG = EXPERIMENT / "config.json"  # 로컬 기본 설정: 실행 비활성화
cfg = read_config(CONFIG)
print("Config:", CONFIG)
print("Execution enabled:", cfg["execution_enabled"])
print("Storage:", cfg["storage_root"])


## 평가 실행
서버에서 수행합니다. CSV와 JSON은 외부 storage_root/runs 아래에 저장됩니다.

In [ ]:
from evaluate_regions import evaluate
cases, folds, comparison = evaluate(cfg)


## 결과 확인

In [ ]:
import pandas as pd
from IPython.display import display
display(folds * 100)
display(pd.DataFrame({
    "metric": ["Dice (%)", "NSD (%)"],
    "paper_5fold": [86.04, 82.91],
    "this_run": [comparison["dice_percent"], comparison["nsd_percent"]],
}))
print("Complete 5-fold:", comparison["complete_5fold"])
print("Reports:", paths(cfg)["run"] / "reports")
print("Limitations:", comparison["limitations"])
